# Ewaluacja apelacji (na przykładzie baseline)

Dwa spojrzenia: **pokrycie** (czy porusza wymagane zagadnienia z `data/eval.json`)
oraz **jakość/forma** (jak u egzaminatora). Działa na zapisanej apelacji — najpierw
wygeneruj ją w `baseline_walkthrough.ipynb`.

## 0. Konfiguracja

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()
os.environ["LLM_MODEL"] = "gpt-5.4-mini"
print("model:", os.environ["LLM_MODEL"], "| klucz z .env:", bool(os.environ.get("LLM_API_KEY")))

model: gpt-5.4-mini | klucz z .env: True


## 1. Wczytaj zapisaną apelację baseline

In [2]:
from src.output import load_latest_appeal, latest_appeal_path

appeal = load_latest_appeal("baseline")
print("Wczytano:", latest_appeal_path("baseline"))
print(f"{len(appeal):,} znaków")

Wczytano: /Users/ewasuknarowska/Projects/WomenInTech/baseline/output/apelacja_baseline_gpt-5.4_2026-06-06_204759.txt
11,707 znaków


## 2. Pokrycie — czy porusza wymagane zagadnienia?

Sędzia-LLM ocenia każde zagadnienie z klucza (na bieżąco).

In [3]:
from src.eval.coverage import evaluate

cov = evaluate(appeal, print_results=True)

[1/12] ✅ 1. W zakresie czynu z art. 178a § 1 k.k. (zarzut pkt I z aktu oskarżenia) istotnym uchybieniem, jakiego dopuścił się Sąd Rejonowy, jest błąd w ustaleniach faktycznych, polegający na dowolnym (bez oparcia w dowodzie na tę okoliczność) ustaleniu, że oskarżony w dniu zdarzenia o godz. 8.30 przyjeżdżając z domu w Kobylanach na cmentarz w Szczyglicach, był nietrzeźwy. Na tę okoliczność nie przeprowadzono w sprawie żadnego dowodu, w szczególności:
a) brak co do tej okoliczności przyznania się oskarżonego,
     → Apelacja faktycznie podnosi zarzut błędu w ustaleniach faktycznych co do tego, czy oskarżony był nietrzeźwy w chwili prowadzenia pojazdu o godz. 8.30. Wprost wskazuje, że brak jest dowodu na tę okoliczność, podkreśla brak przyznania się oskarżonego, brak możliwości retrospektywnego ustalenia stężenia alkoholu oraz twierdzi, że oskarżony spożył alkohol dopiero po zatrzymaniu pojazdu. To odpowiada istocie zagadnienia z klucza.

[2/12] ✅ 1. W zakresie czynu z art. 178a § 1 k.k.

## 3. Jakość / forma (ocena egzaminatora)

Ocena „miękka” wg kryteriów egzaminu radcowskiego (wymogi formalne / zastosowanie
przepisów / poprawność rozwiązania), skala 2–6.

> Sędzią jakości jest **mocny model** (`gpt-5.4`) — tani model nie wyłapuje błędów
> formalnych. To 1 wywołanie, niezależne od modelu z konfiguracji.

In [4]:
from src.eval.quality import evaluate_quality

q = evaluate_quality(appeal)
print(q.reasoning, "\n")
print(f"  wymogi formalne:            {q.wymogi_formalne}/6")
print(f"  zastosowanie/interpretacja: {q.zastosowanie_i_interpretacja}/6")
print(f"  poprawność rozwiązania:     {q.poprawnosc_rozwiazania}/6")
print(f"  -- średnia:                 {q.srednia:.2f}/6")

Pismo jest napisane na ogół poprawnie i w sposób uporządkowany. Na plus należy zaliczyć prawidłowe oznaczenie sądu odwoławczego i wniesienie apelacji za pośrednictwem sądu I instancji, wskazanie sygnatury, oskarżonego i obrońcy, określenie zakresu zaskarżenia oraz wyodrębnienie zarzutów, wniosków i uzasadnienia. Autor zachowuje przejrzystą strukturę: osobno omawia czyn z art. 178a § 1 k.k., osobno sankcje oraz osobno czyn z art. 288 § 1 k.k. Argumentacja jest komunikatywna, oparta na konkretnych elementach materiału dowodowego, a zarzuty są powiązane z wnioskami odwoławczymi.

Mocną stroną pisma jest część dotycząca podważenia skazania z art. 178a § 1 k.k. Trafnie wyeksponowano problem zasadniczy: nie sam stan nietrzeźwości w chwili interwencji, lecz brak pewności co do stanu oskarżonego w chwili prowadzenia pojazdu. Prawidłowo odwołano się do zarzutu błędu w ustaleniach faktycznych oraz do art. 7 k.p.k., art. 410 k.p.k. i art. 5 § 2 k.p.k. Uzasadnienie pokazuje, że autor rozumie stand